In [2]:
import os
import time
import requests
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib as mpl
import matplotlib.pyplot as plt
import StringAPI as string
import gseapy as gp
from pathlib import Path
from io import StringIO
from matplotlib.lines import Line2D


# Illustrator settings
mpl.rcParams["font.family"] = "Arial"
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"


# Paths
#main_dir = '/Users/ga0028vi/Documents/MCcharactertization/ROSMAP/Top_sets/Set309'
main_dir = 'C:/Users/gabri/Documents/Data_projects/MCcharactertization/ROSMAP/Top_sets/Set309'

ref_dir = Path(main_dir, "_set_data")

out_dir = Path(main_dir, "STRING_AGM_analysis/GSEAonAGMsnewnames")
#out_dir = Path(main_dir, "STRING_AGM_analysis/GSEAonAGMs")
out_dir.mkdir(exist_ok=True)


# Read gene list data
#df = pd.read_csv(Path(main_dir, "_set_data/PLSsig_genelists_long.csv"))
#df = pd.read_csv(Path('/Users/ga0028vi/Documents/MCcharactertization/ROSMAP/Top_sets/Set309/STRING_AGM_analysis/Annotations/PLSgenelist_long_HGNCcorrected.csv'), index_col=False)
df = pd.read_csv(Path('C:/Users/gabri/Documents/Data_projects/MCcharactertization/ROSMAP/Top_sets/Set309/STRING_AGM_analysis/Annotations/PLSgenelist_long_HGNCcorrected.csv'), index_col=False)
# Enrichr libraries
libraries = [
    "KEGG_2021_Human",
    "GO_Biological_Process_2023",
    "GO_Molecular_Function_2023"
]

FDR_filter = 0.05
top_n_to_save = 20
sleep_time = 2


In [3]:
df['gene']=df['gene_renamed']

In [ ]:
# Run ORA for each whole AGM
all_agm_enrichr_results = []

for module_name in df["AGM_ID"].unique():

    print("\n" + "=" * 80)
    print(f"Running whole-AGM ORA: {module_name}")
    print("=" * 80)

    # Get cell type from AGM name
    celltype = module_name.split("_")[0]

    # Read matching background genes
    ref_genes = pd.read_csv(
        Path(ref_dir, f"{celltype}_ref_genes.txt"),
        header=None
    )[0]

    ref_genes = (
        ref_genes
        .dropna()
        .astype(str)
        .str.strip()
        .unique()
        .tolist()
    )

    # Get genes for this AGM
    genes = (
        df.loc[df["AGM_ID"] == module_name, "gene"]
        .dropna()
        .astype(str)
        .str.strip()
        .unique()
        .tolist()
    )

    print(f"{module_name}: {len(genes)} AGM genes | {len(ref_genes)} background genes")

    # Optional: skip very tiny AGMs
    if len(genes) < 3:
        print(f"Skipping {module_name}: fewer than 3 genes")
        continue

    try:
        enr = gp.enrichr(
            gene_list=genes,
            gene_sets=libraries,
            organism="human",
            background=ref_genes,
            outdir=None,
            cutoff=FDR_filter
        )

        res = enr.results.copy()

        # Keep only significant terms
        res = res.loc[res["Adjusted P-value"] < FDR_filter].copy()

        if res.empty:
            print(f"No significant terms for {module_name}")
            continue

        # Keep top N terms per database
        res = (
            res
            .sort_values(["Gene_set", "Adjusted P-value", "P-value"])
            .groupby("Gene_set", as_index=False)
            .head(top_n_to_save)
        )

        # Add metadata
        res.insert(0, "AGM_ID", module_name)
        res.insert(1, "AGM_size", len(genes))
        res.insert(2, "n_background_genes", len(ref_genes))

        # Save individual AGM result
        res.to_csv(
            out_dir / f"{module_name}_whole_AGM_enrichr.csv",
            index=False
        )

        all_agm_enrichr_results.append(res)

        print(f"Saved ORA results for {module_name}")

    except Exception as e:
        print(f"Failed {module_name}: {e}")

    time.sleep(sleep_time)


# Save combined result
if all_agm_enrichr_results:
    all_agm_enrichr_df = pd.concat(
        all_agm_enrichr_results,
        ignore_index=True
    )
else:
    all_agm_enrichr_df = pd.DataFrame()

all_agm_enrichr_df.to_csv(
    out_dir / "ALL_whole_AGM_enrichr_summary.csv",
    index=False
)

print("Finished all whole-AGM ORA runs.")


Running whole-AGM ORA: Astrocytes_AstDPP10_Cluster_6
Astrocytes_AstDPP10_Cluster_6: 87 AGM genes | 2213 background genes
No significant terms for Astrocytes_AstDPP10_Cluster_6

Running whole-AGM ORA: Excitatory_ExcL34RORBCUX2_Cluster_6
Excitatory_ExcL34RORBCUX2_Cluster_6: 91 AGM genes | 2455 background genes
Saved ORA results for Excitatory_ExcL34RORBCUX2_Cluster_6

Running whole-AGM ORA: Excitatory_ExcL35RORBPLCH1_Cluster_14
Excitatory_ExcL35RORBPLCH1_Cluster_14: 191 AGM genes | 2455 background genes
Saved ORA results for Excitatory_ExcL35RORBPLCH1_Cluster_14

Running whole-AGM ORA: Excitatory_ExcL45RORBGABRG1_Cluster_0
Excitatory_ExcL45RORBGABRG1_Cluster_0: 127 AGM genes | 2455 background genes
No significant terms for Excitatory_ExcL45RORBGABRG1_Cluster_0

Running whole-AGM ORA: Excitatory_ExcL45RORBGABRG1_Cluster_13
Excitatory_ExcL45RORBGABRG1_Cluster_13: 84 AGM genes | 2455 background genes
Saved ORA results for Excitatory_ExcL45RORBGABRG1_Cluster_13

Running whole-AGM ORA: Excit